In [ ]:
from ultralytics import YOLO
import torch

In [ ]:


# Kiểm tra GPU có khả dụng không
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

# Load a model
model = YOLO("model/detect/yolo11n.pt")  # load a pretrained model (recommended for training)

# Train the model với cấu hình GPU
results = model.train(
    data="data/yolov11_detection_dataset_cccd/dataset.yaml",
    epochs=350,
    imgsz=640,
    device=0,  # Sử dụng GPU đầu tiên (0), hoặc 'cpu' nếu muốn dùng CPU
    batch=16,  # Điều chỉnh batch size phù hợp với GPU memory
    workers=4,  # Số worker threads
    verbose=True,
    amp=False,
)

In [ ]:
model = YOLO("runs/detect/train/weights/best.pt")  # load a custom model

In [ ]:
# batch_size 16 epoch 100
# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list contains map50-95 of each category

In [ ]:
# Run batched inference on a list of images
results = model(
    ["data/yolov11_detection_dataset_cccd/val/images/294_1_VTM_20240925103911_681.jpg",
    "data/yolov11_detection_dataset_cccd/val/images/603_3_VTM_20240718101645_545.jpg",
    "data/yolov11_detection_dataset_cccd/val/images/DEP_297_1_VTM_20250422210623_385.jpg",
    "data/yolov11_detection_dataset_cccd/val/images/DEP_304_1_VTM_20250210154033_303.jpg",
    "data/yolov11_detection_dataset_cccd/val/images/DEP_706_3_VTM_20240705185346_160.jpg"],
    # stream=True
)  # return a generator of Results objects

# Process results generator
for i,result in enumerate(results):
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    # result.show()  # display to screen
    result.save(filename=f"text_det_cccd_result{i}.jpg")  # save to disk